# Log Anomaly Detection (HDFS Dataset): LLM Few-Shot + CoT Baseline

**Experiment:** E02 — LLM-based anomaly detection using few-shot prompting with Chain-of-Thought reasoning.  
**Model:** `llama-3.3-70b-versatile` via Groq API.  
**Evaluation model:** `qwen/qwen3-32b` via Groq API (LLM-as-judge).  

In [1]:
# =============================================================================
# CELL 1 — PACKAGE INSTALLATION
# =============================================================================

import subprocess
import sys


def install_packages(packages):
    for pkg in packages:
        print(f"Installing: {pkg}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", pkg, "-q"],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            print(f"  WARNING: may have failed -- {result.stderr[:200]}")
        else:
            print("  OK")


install_packages([
    "pandas",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "groq",
    "tqdm",
])
print("\nAll packages ready.")

Installing: pandas
  OK
Installing: numpy
  OK
Installing: scikit-learn
  OK
Installing: matplotlib
  OK
Installing: seaborn
  OK
Installing: groq
  OK
Installing: tqdm
  OK

All packages ready.


In [3]:
# =============================================================================
# CELL 2 — IMPORTS, CONFIGURATION, DATA LOADING, LLM INFERENCE,
#           STANDARD METRICS, LLM-AS-JUDGE EVALUATION, AND VISUALISATION
# =============================================================================

# -- Standard library imports -------------------------------------------------
import json
import os
import re
import time
import warnings
from typing import Any, Dict, List, Optional, Tuple

# -- Third-party imports ------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from groq import Groq
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from tqdm import tqdm

warnings.filterwarnings("ignore")

# =============================================================================
# SECTION 1 — CONFIGURATION
# =============================================================================

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# -- Dataset ------------------------------------------------------------------
DATASET = "BGL"

# -- Groq API key — read from src/Keys/groq_key.txt; fall back to env var ----
_GROQ_KEY_FILE = os.path.join(os.path.dirname(os.getcwd()), "Keys", "groq_key.txt")
if os.path.isfile(_GROQ_KEY_FILE):
    with open(_GROQ_KEY_FILE, "r") as _f:
        GROQ_API_KEY = _f.read().strip()
else:
    GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
    if not GROQ_API_KEY:
        raise FileNotFoundError(
            f"Groq key file not found at {_GROQ_KEY_FILE} and "
            "GROQ_API_KEY environment variable is not set."
        )

# -- Model names (configurable) -----------------------------------------------
LLM_MODEL  = "llama-3.3-70b-versatile"  # anomaly detection / generation model
EVAL_MODEL = "qwen/qwen3-32b"           # LLM-as-judge evaluation model

# -- Few-shot prompt configuration --------------------------------------------
N_FEW_SHOT_NORMAL    = 3  # normal examples included in system prompt
N_FEW_SHOT_ANOMALOUS = 3  # anomalous examples included in system prompt

# -- Groq API call configuration ----------------------------------------------
MAX_RETRIES      = 3
RETRY_DELAY_S    = 5    # seconds between retries on transient errors
REQUEST_DELAY_S  = 0.5  # seconds between successive calls (rate-limit buffer)
LLM_MAX_TOKENS   = 600
LLM_TEMPERATURE  = 0.0
EVAL_MAX_TOKENS  = 150

# -- Split ratios — must match E01_C to keep test set size identical ----------
TRAIN_RATIO = 0.60
VAL_RATIO   = 0.20
# test = remaining 20% of normals + all anomalies (same as E01_C: ~2303 samples)

# -- LLM-as-judge evaluation subset size -------------------------------------
# Calling qwen/qwen3-32b for all 2300+ test samples is expensive.
# Set to None to evaluate the entire test set.
EVAL_SUBSET_SIZE = 100

# -- Dataset-specific configurations (dataset-agnostic loading logic below) --
DATASET_CONFIGS: Dict[str, Dict] = {
    "HDFS": {
        "traces_path":    "../../Datasets/HDFS/Full_HDFS_v1/preprocessed/Event_traces.csv",
        "templates_path": "../../Datasets/HDFS/Full_HDFS_v1/preprocessed/HDFS.log_templates.csv",
        "label_col":    "Label",
        "normal_value": "Success",
        "features_col": "Features",
        # Sampling caps — same as E01_C for identical dataset fingerprint
        "normal_sample_cap":  10000,
        "anomaly_type_col":   "Type",
        "anomaly_n_per_type": 20,
    },
    "BGL": {
        "structured_log_path": "../../Datasets/BGL/Sample/BGL_2k.log_structured.csv",
        "templates_path":      "../../Datasets/BGL/Sample/BGL_2k.log_templates.csv",
        "label_col":     "Label",
        "normal_value":  "-",
        "event_id_col":  "EventId",
        "component_col": "Component",
        "level_col":     "Level",
        "content_col":   "Content",
        "template_col":  "EventTemplate",
        "normal_sample_cap":  None,
        "anomaly_type_col":   None,
        "anomaly_n_per_type": None,
    },
}

print(f"  Dataset          : {DATASET}")
print(f"  LLM model        : {LLM_MODEL}")
print(f"  Eval model       : {EVAL_MODEL}")
print(f"  Few-shot normal  : {N_FEW_SHOT_NORMAL}")
print(f"  Few-shot anomaly : {N_FEW_SHOT_ANOMALOUS}")
print(f"  Eval subset size : {EVAL_SUBSET_SIZE}")
print(f"  Train ratio      : {TRAIN_RATIO}")
print(f"  Val ratio        : {VAL_RATIO}")
print(f"  Test normals     : remaining {1.0 - TRAIN_RATIO - VAL_RATIO:.0%} of normals")
print(f"  Test anomalies   : ALL available (minus few-shot pool)")
print(f"  Groq key source  : {'key file' if os.path.isfile(_GROQ_KEY_FILE) else 'env var'}")

# =============================================================================
# SECTION 2 — DATA LOADING (DATASET-AGNOSTIC)
#
# Output contract (identical for every dataset):
#   log_text     (str)  — enriched text fed to the LLM
#   is_normal    (bool) — True if the sample is labelled Normal
#   binary_label (int)  — 0 = Normal, 1 = Anomalous
# =============================================================================

def _apply_sampling(
    df: pd.DataFrame,
    config: Dict,
    random_seed: int = 42,
) -> pd.DataFrame:
    """Apply per-dataset sampling caps.  No-op when all caps are None."""
    normal_cap    = config.get("normal_sample_cap")
    anom_type_col = config.get("anomaly_type_col")
    n_per_type    = config.get("anomaly_n_per_type")
    anom_cap      = config.get("anomaly_sample_cap")

    df_n = df[df["is_normal"]].copy()
    df_a = df[~df["is_normal"]].copy()

    if normal_cap is not None and len(df_n) > normal_cap:
        df_n = df_n.sample(n=normal_cap, random_state=random_seed)

    if anom_type_col and n_per_type:
        parts = [
            grp.sample(n=min(n_per_type, len(grp)), random_state=random_seed)
            for _, grp in df_a.groupby(anom_type_col)
            if len(grp) > 0
        ]
        df_a = pd.concat(parts) if parts else df_a
    elif anom_cap is not None and len(df_a) > anom_cap:
        df_a = df_a.sample(n=anom_cap, random_state=random_seed)

    return pd.concat([df_n, df_a], ignore_index=True)


def _clean_template(text: str) -> str:
    """Remove Drain wildcards [*] and collapse whitespace."""
    cleaned = re.sub(r'\[\*\]', '', text)
    return re.sub(r'\s+', ' ', cleaned).strip()


def build_hdfs_log_text(
    row: pd.Series,
    config: Dict,
    template_lookup: Dict,
) -> str:
    """Build block-trace text from an ordered HDFS event-ID sequence."""
    features_str = str(row.get(config["features_col"], "[]"))
    event_ids    = re.findall(r'E\d+', features_str)
    seen, unique_templates = set(), []
    for eid in event_ids:
        if eid not in seen:
            seen.add(eid)
            tmpl = template_lookup.get(eid, eid)
            unique_templates.append(_clean_template(tmpl))
    return "HDFS Block Trace | " + " -> ".join(unique_templates)


def build_bgl_log_text(row: pd.Series, config: Dict) -> str:
    """Build enriched text for a single BGL log line."""
    component = str(row.get(config["component_col"], "")).strip()
    level     = str(row.get(config["level_col"],     "")).strip()
    content   = str(row.get(config["content_col"],   "")).strip()
    template  = str(row.get(config["template_col"],  "")).strip()
    return f"[{component}] [{level}] {content} | Template: {template}"


def load_hdfs_data(config: Dict) -> pd.DataFrame:
    """Load HDFS Event_traces.csv and build block-level log_text."""
    print("  Loading HDFS block traces ...")
    df_traces    = pd.read_csv(config["traces_path"])
    df_templates = pd.read_csv(config["templates_path"])
    print(f"    Block traces: {len(df_traces)}, Templates: {len(df_templates)}")

    template_lookup = dict(
        zip(df_templates["EventId"], df_templates["EventTemplate"])
    )
    df_traces["is_normal"]    = (
        df_traces[config["label_col"]] == config["normal_value"]
    )
    df_traces["binary_label"] = (~df_traces["is_normal"]).astype(int)
    df_traces = _apply_sampling(df_traces, config)

    n_s = int(df_traces["is_normal"].sum())
    n_a = int((~df_traces["is_normal"]).sum())
    print(f"    After sampling : {len(df_traces)} blocks ({n_s} normal, {n_a} anomalous)")

    df_traces["log_text"] = df_traces.apply(
        lambda row: build_hdfs_log_text(row, config, template_lookup), axis=1
    )
    return df_traces


def load_bgl_data(config: Dict) -> pd.DataFrame:
    """Load BGL structured log CSV, merge templates, build log_text."""
    print("  Loading BGL structured log ...")
    df_logs      = pd.read_csv(config["structured_log_path"])
    df_templates = pd.read_csv(config["templates_path"])
    print(f"    Rows: {len(df_logs)}, Templates: {len(df_templates)}")

    df = df_logs.merge(
        df_templates,
        on=config["event_id_col"],
        how="left",
        suffixes=("", "_tmpl"),
    )
    df["is_normal"]    = df[config["label_col"]] == config["normal_value"]
    df["binary_label"] = (~df["is_normal"]).astype(int)
    df = _apply_sampling(df, config)
    df["log_text"] = df.apply(
        lambda row: build_bgl_log_text(row, config), axis=1
    )
    return df


def load_data(dataset_name: str, configs: Dict) -> pd.DataFrame:
    """Dispatch to dataset-specific loader.

    Returns DataFrame guaranteed to have:
        log_text (str), is_normal (bool), binary_label (int).
    """
    if dataset_name not in configs:
        raise ValueError(
            f"Unknown dataset '{dataset_name}'. Available: {list(configs.keys())}"
        )
    if dataset_name == "HDFS":
        return load_hdfs_data(configs[dataset_name])
    elif dataset_name == "BGL":
        return load_bgl_data(configs[dataset_name])


print(f"Loading dataset: {DATASET}")
df = load_data(DATASET, DATASET_CONFIGS)

n_normal    = int(df["is_normal"].sum())
n_anomalous = int((~df["is_normal"]).sum())
print(f"\n  Total samples : {len(df)}")
print(f"  Normal        : {n_normal}  ({n_normal / len(df) * 100:.1f}%)")
print(f"  Anomalous     : {n_anomalous}  ({n_anomalous / len(df) * 100:.1f}%)")
print(f"\n  Sample log_text:\n    {df['log_text'].iloc[0]}")

# =============================================================================
# SECTION 3 — TRAIN / VALIDATION / TEST SPLIT
#
# Mirrors E01_C exactly:
#   Train   : 60% of normals  (OC-SVM training — unused here but kept for parity)
#   Val     : 20% of normals  (unused here but kept for parity)
#   Test    : 20% of normals  + all remaining anomalies  (~2303 samples)
#
# N_FEW_SHOT_ANOMALOUS examples are carved out of the anomaly pool BEFORE
# constructing the test set to avoid any data leakage into reported metrics.
# =============================================================================

def create_splits_with_few_shot(
    df: pd.DataFrame,
    train_ratio: float = 0.60,
    val_ratio: float = 0.20,
    n_few_shot_anomaly: int = 3,
    random_seed: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return (train_df, val_df, test_df, few_shot_anomaly_df).

    Few-shot anomaly examples are excluded from the test evaluation pool
    to prevent data leakage from prompt context into reported metrics.
    """
    assert train_ratio + val_ratio <= 1.0, "train_ratio + val_ratio must not exceed 1.0"

    df_normal = (
        df[df["is_normal"]]
        .sample(frac=1, random_state=random_seed)
        .reset_index(drop=True)
    )
    df_anomalous = (
        df[~df["is_normal"]]
        .sample(frac=1, random_state=random_seed)
        .reset_index(drop=True)
    )

    # Reserve few-shot anomaly examples before building the test set
    n_fs              = min(n_few_shot_anomaly, len(df_anomalous))
    few_shot_anom_df  = df_anomalous.iloc[:n_fs].reset_index(drop=True)
    df_anomalous_test = df_anomalous.iloc[n_fs:].reset_index(drop=True)

    n_total = len(df_normal)
    n_train = int(n_total * train_ratio)
    n_val   = int(n_total * val_ratio)

    train_df    = df_normal.iloc[:n_train].reset_index(drop=True)
    val_df      = df_normal.iloc[n_train: n_train + n_val].reset_index(drop=True)
    test_normal = df_normal.iloc[n_train + n_val:].reset_index(drop=True)

    test_df = (
        pd.concat([test_normal, df_anomalous_test], ignore_index=True)
        .sample(frac=1, random_state=random_seed)
        .reset_index(drop=True)
    )
    return train_df, val_df, test_df, few_shot_anom_df


print("Creating train / validation / test splits ...")
train_df, val_df, test_df, few_shot_anom_df = create_splits_with_few_shot(
    df,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    n_few_shot_anomaly=N_FEW_SHOT_ANOMALOUS,
    random_seed=RANDOM_SEED,
)

n_test_normal    = int(test_df["is_normal"].sum())
n_test_anomalous = int((~test_df["is_normal"]).sum())

print(f"\n  Train  : {len(train_df):>6}  normal only")
print(f"  Val    : {len(val_df):>6}  normal only")
print(f"  Test   : {len(test_df):>6}  total")
print(f"           {n_test_normal:>6}  normal     ({n_test_normal / len(test_df) * 100:.1f}%)")
print(f"           {n_test_anomalous:>6}  anomalous  ({n_test_anomalous / len(test_df) * 100:.1f}%)")
print(f"  Few-shot anomaly pool : {len(few_shot_anom_df)} samples (excluded from test evaluation)")

# =============================================================================
# SECTION 4 — FEW-SHOT EXAMPLE SELECTION
# =============================================================================

# Normal examples come from the training set (no leakage risk)
few_shot_normal_df = train_df.sample(
    n=min(N_FEW_SHOT_NORMAL, len(train_df)),
    random_state=RANDOM_SEED,
)

print(f"\nFew-shot examples selected:")
print(f"  Normal    : {len(few_shot_normal_df)} (from train set)")
print(f"  Anomalous : {len(few_shot_anom_df)} (carved out before test split)")

# =============================================================================
# SECTION 5 — PROMPT CONSTRUCTION (FEW-SHOT + CHAIN-OF-THOUGHT)
#
# Each few-shot example follows the CoT pattern:
#   1. Observe the event sequence
#   2. Identify deviations (or lack thereof)
#   3. Trace the causal chain
#   4. Output structured JSON conclusion
#
# The templates use {{ }} to escape literal braces inside an f-string context.
# =============================================================================

_COT_NORMAL_TEMPLATE = """\
Log: {log_text}
Analysis:
- The event sequence follows the expected HDFS block lifecycle \
(request -> transfer -> acknowledge -> close).
- No error, exception, retry, or out-of-order events are present.
- All pipeline stages complete with standard handshake patterns.
- Causal chain: block requested -> DataNode accepted write pipeline -> \
data transferred in packets -> block acknowledged -> pipeline closed normally.
Conclusion:
{{
  "label": "Normal",
  "confidence": 0.95,
  "explanation": {{
    "system_impact_risk": "No impact. Block lifecycle completed without disruption to \
HDFS availability or data integrity.",
    "root_cause": "Not applicable — no anomaly detected.",
    "causal_chain": "Block requested by client -> NameNode allocated DataNode -> \
write pipeline established -> data packets transferred -> checksum verified -> \
block acknowledged -> pipeline closed successfully.",
    "supporting_evidence": "Event sequence contains only expected pipeline events \
with no error, exception, or retry events."
  }}
}}"""

_COT_ANOMALOUS_TEMPLATE = """\
Log: {log_text}
Analysis:
- The event sequence deviates from the normal HDFS block pipeline pattern.
- Error or failure events are present, indicating a mid-pipeline disruption.
- Unexpected event ordering or missing acknowledgement events are observed.
- Causal chain: block write initiated -> network/disk fault encountered -> \
pipeline stage aborted -> DataNode reported failure -> NameNode notified for re-replication.
Conclusion:
{{
  "label": "Anomalous",
  "confidence": 0.92,
  "explanation": {{
    "system_impact_risk": "Data block may be under-replicated or lost. \
Client write will fail or stall. Risk of data unavailability if replication \
factor drops below the configured minimum (default: 3).",
    "root_cause": "DataNode failure or network disruption during block write \
pipeline execution caused the pipeline to break before acknowledgement.",
    "causal_chain": "Block write initiated -> intermediate DataNode encountered \
error -> pipeline stage aborted -> client received write exception -> \
NameNode scheduled re-replication of under-replicated block.",
    "supporting_evidence": "Error and exception events present in the event \
sequence indicating pipeline failure before block acknowledgement was received."
  }}
}}"""


def build_system_prompt(
    few_shot_normal: pd.DataFrame,
    few_shot_anomalous: pd.DataFrame,
) -> str:
    """Build the system prompt with role definition and few-shot CoT examples."""
    normal_examples = "\n\n".join(
        f"Example {i + 1} (Normal):\n"
        + _COT_NORMAL_TEMPLATE.format(log_text=row["log_text"])
        for i, (_, row) in enumerate(few_shot_normal.iterrows())
    )
    anomalous_examples = "\n\n".join(
        f"Example {i + 1} (Anomalous):\n"
        + _COT_ANOMALOUS_TEMPLATE.format(log_text=row["log_text"])
        for i, (_, row) in enumerate(few_shot_anomalous.iterrows())
    )

    return (
        "You are a senior Site Reliability Engineer (SRE) specialising in distributed "
        "storage systems. Your task is to analyse HDFS (Hadoop Distributed File System) "
        "block operation log traces and determine whether each trace represents a Normal "
        "operation or an Anomalous condition.\n\n"
        "For each log trace you will:\n"
        "1. Identify the event sequence and assess whether it follows the expected HDFS "
        "block lifecycle.\n"
        "2. Detect deviations: error events, unexpected transitions, missing acknowledgements, "
        "or retries.\n"
        "3. Reason step-by-step (Chain-of-Thought) before producing a structured JSON "
        "conclusion.\n\n"
        'Your response MUST be valid JSON with exactly this schema:\n'
        '{\n'
        '  "label": "Normal" | "Anomalous",\n'
        '  "confidence": <float 0.0-1.0, probability of being Anomalous>,\n'
        '  "explanation": {\n'
        '    "system_impact_risk": "<impact on HDFS availability, data integrity, and client operations>",\n'
        '    "root_cause": "<underlying technical cause, or Not applicable if Normal>",\n'
        '    "causal_chain": "<ordered sequence of events leading to the outcome>",\n'
        '    "supporting_evidence": "<specific events or patterns in the log that support the conclusion>"\n'
        '  }\n'
        '}\n\n'
        "--- FEW-SHOT EXAMPLES (NORMAL) ---\n\n"
        + normal_examples
        + "\n\n--- FEW-SHOT EXAMPLES (ANOMALOUS) ---\n\n"
        + anomalous_examples
        + "\n\n--- END OF EXAMPLES ---\n\n"
        "Now analyse the log trace provided in the user message. "
        "Respond ONLY with valid JSON."
    )


def build_user_prompt(log_text: str) -> str:
    """Build the per-sample user message."""
    return f"Log: {log_text}"


SYSTEM_PROMPT = build_system_prompt(few_shot_normal_df, few_shot_anom_df)
print(f"System prompt length : {len(SYSTEM_PROMPT)} characters")

# =============================================================================
# SECTION 6 — GROQ CLIENT AND INFERENCE HELPERS
# =============================================================================

groq_client = Groq(api_key=GROQ_API_KEY)


def call_groq(
    system_prompt: str,
    user_message: str,
    model: str,
    max_tokens: int = LLM_MAX_TOKENS,
    temperature: float = LLM_TEMPERATURE,
    max_retries: int = MAX_RETRIES,
    retry_delay: float = RETRY_DELAY_S,
) -> Optional[str]:
    """Call the Groq chat completions API with retry logic.

    Returns raw response text, or None if all retries are exhausted.
    """
    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_message},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return response.choices[0].message.content.strip()
        except Exception as exc:
            wait = retry_delay * (attempt + 1)
            print(
                f"  API error (attempt {attempt + 1}/{max_retries}): "
                f"{exc}. Retrying in {wait}s ..."
            )
            time.sleep(wait)
    return None


def parse_llm_response(raw: Optional[str]) -> Dict[str, Any]:
    """Extract and validate JSON from LLM response text.

    Strips markdown code fences, extracts the first JSON object,
    and fills in safe defaults on any parse failure.
    """
    _default_explanation = {
        "system_impact_risk": "Parse error",
        "root_cause":         "Parse error",
        "causal_chain":       "Parse error",
        "supporting_evidence": "Parse error",
    }
    _default = {
        "label":       "Normal",
        "confidence":  0.5,
        "explanation": _default_explanation,
        "parse_error": True,
    }
    if raw is None:
        return _default.copy()

    try:
        cleaned = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`")
        match   = re.search(r'\{.*\}', cleaned, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
            parsed.setdefault("parse_error", False)
            parsed.setdefault("explanation", _default_explanation)
            return parsed
    except (json.JSONDecodeError, ValueError):
        pass

    return _default.copy()


# =============================================================================
# SECTION 7 — RUN LLM INFERENCE ON TEST SET
# =============================================================================

print(f"\nRunning LLM inference on {len(test_df)} test samples ...")
print(f"  Model       : {LLM_MODEL}")
print(f"  Temperature : {LLM_TEMPERATURE}")
print(f"  Max tokens  : {LLM_MAX_TOKENS}")
print("  Applying rate-limit delay between calls.\n")

results: List[Dict[str, Any]] = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="LLM Inference"):
    user_msg = build_user_prompt(row["log_text"])
    raw_resp = call_groq(
        system_prompt=SYSTEM_PROMPT,
        user_message=user_msg,
        model=LLM_MODEL,
    )
    parsed                 = parse_llm_response(raw_resp)
    parsed["true_label"]   = int(row["binary_label"])
    parsed["log_text"]     = row["log_text"]
    parsed["raw_response"] = raw_resp
    results.append(parsed)
    time.sleep(REQUEST_DELAY_S)

results_df = pd.DataFrame(results)

# Map label string to binary integer (1 = Anomalous)
results_df["pred_label"] = results_df["label"].apply(
    lambda x: 1 if str(x).strip().lower() == "anomalous" else 0
)

# Use LLM confidence score as continuous predictor for ROC/PR curves
results_df["confidence_score"] = pd.to_numeric(
    results_df["confidence"], errors="coerce"
).fillna(results_df["pred_label"].astype(float))

n_parse_errors = int(results_df["parse_error"].sum())
print(f"\n  Inference complete.")
print(f"  Parse errors : {n_parse_errors} / {len(results_df)}")

# =============================================================================
# SECTION 8 — STANDARD ANOMALY DETECTION METRICS
# =============================================================================

def evaluate_predictions(
    y_true: List[int],
    y_pred: List[int],
    dataset_name: str,
    model_name: str = "LLM Few-Shot + CoT",
) -> Dict[str, Any]:
    """Compute and print standard one-class anomaly detection metrics.

    Identical interface to E01_C for direct comparison of results.
    """
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall    = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1        = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    cm        = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    n_normal_t    = y_true.count(0)
    n_anomalous_t = y_true.count(1)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    print("\n" + "=" * 64)
    print(f"  EVALUATION  --  {model_name}  [{dataset_name}]")
    print("=" * 64)
    print(f"  Test set size          : {len(y_true)}")
    print(
        f"    Normal   (neg)       : {n_normal_t}  "
        f"({n_normal_t / len(y_true) * 100:.1f}%)"
    )
    print(
        f"    Anomalous (pos)      : {n_anomalous_t}  "
        f"({n_anomalous_t / len(y_true) * 100:.1f}%)"
    )
    print("-" * 64)
    print(f"  TP (caught anomalies)  : {tp}")
    print(f"  TN (correct normals)   : {tn}")
    print(f"  FP (false alarms)      : {fp}")
    print(f"  FN (missed anomalies)  : {fn}")
    print("-" * 64)
    print(f"  Accuracy               : {accuracy:.4f}  ({accuracy * 100:.2f}%)")
    print(f"  Precision              : {precision:.4f}")
    print(f"  Recall                 : {recall:.4f}")
    print(f"  F1-Score               : {f1:.4f}")
    print(f"  FPR (false alarm rate) : {fpr:.4f}")
    print(f"  FNR (miss rate)        : {fnr:.4f}")
    print("=" * 64)

    return dict(
        accuracy=accuracy, precision=precision, recall=recall, f1=f1,
        tp=int(tp), tn=int(tn), fp=int(fp), fn=int(fn),
        cm=cm, fpr=fpr, fnr=fnr,
    )


y_true      = results_df["true_label"].tolist()
y_pred      = results_df["pred_label"].tolist()
conf_scores = results_df["confidence_score"].tolist()

metrics = evaluate_predictions(y_true, y_pred, DATASET)

# AUC-ROC using LLM confidence score as the continuous ranking signal
auroc = roc_auc_score(y_true, conf_scores)
fpr_roc, tpr_roc, roc_thresholds = roc_curve(y_true, conf_scores)
pr_precisions, pr_recalls, _     = precision_recall_curve(y_true, conf_scores)
metrics["auroc"] = auroc
print(f"  AUC-ROC                : {auroc:.4f}")

# =============================================================================
# SECTION 9 — LLM-AS-JUDGE EVALUATION METRICS
#
# Evaluates LLM responses using EVAL_MODEL (qwen/qwen3-32b) as judge.
# Five dimensions, each scored [0, 1]:
#   Faithfulness          — explanation claims are grounded in the log
#   Answer Relevance      — response addresses the anomaly detection task
#   Role Appropriateness  — language/depth is suitable for SRE or DevOps
#   Completeness          — all four explanation fields present and substantive
#
# Evaluated on EVAL_SUBSET_SIZE sampled test cases to control API cost.
# =============================================================================

_EVAL_SYSTEM_PROMPT = (
    "You are an objective evaluator of AI-generated log analysis responses. "
    "Score the provided response on the specified dimension. "
    'Respond ONLY with valid JSON: {"score": <float 0.0-1.0>, "reason": "<one sentence>"}'
)


def _build_faithfulness_prompt(log_text: str, explanation: Dict) -> str:
    return (
        f"Log trace: {log_text}\n\n"
        f"AI explanation: {json.dumps(explanation)}\n\n"
        "Score FAITHFULNESS (0-1): Do the claims in the explanation accurately reflect "
        "events or patterns present in the log trace? "
        "1.0 = fully grounded in log evidence, 0.0 = hallucinated or contradicts the log."
    )


def _build_answer_relevance_prompt(log_text: str, full_response: Dict) -> str:
    return (
        f"Log trace: {log_text}\n\n"
        f"AI response: {json.dumps(full_response)}\n\n"
        "Score ANSWER RELEVANCE (0-1): Does the response directly and completely address "
        "the anomaly detection task (identifying whether the log is normal or anomalous "
        "and explaining why)? "
        "1.0 = fully on-task, 0.0 = completely off-topic or missing the detection outcome."
    )


def _build_role_appropriateness_prompt(full_response: Dict, role: str) -> str:
    _role_desc = {
        "SRE": (
            "Site Reliability Engineer who needs actionable, technically precise, "
            "incident-focused analysis with clear severity and system impact statements."
        ),
        "DevOps": (
            "DevOps Engineer who needs operational clarity, root cause identification, "
            "and pipeline/deployment-relevant insights at an intermediate technical level."
        ),
    }
    return (
        f"AI response: {json.dumps(full_response)}\n\n"
        f"Evaluate ROLE APPROPRIATENESS for a {role} "
        f"({_role_desc.get(role, role)}).\n"
        f"Score (0-1): Does the language, depth, and focus of the response match what "
        f"a {role} would need to act on this alert? "
        "1.0 = perfectly tailored for the role, 0.0 = completely inappropriate."
    )


def _build_completeness_prompt(full_response: Dict) -> str:
    return (
        f"AI response: {json.dumps(full_response)}\n\n"
        "Score COMPLETENESS (0-1): Does the explanation include all four required fields "
        "(system_impact_risk, root_cause, causal_chain, supporting_evidence) with "
        "substantive, non-placeholder content? "
        "1.0 = all four fields complete and informative, 0.0 = all fields absent or empty."
    )


def score_with_judge(
    prompt: str,
    eval_model: str = EVAL_MODEL,
) -> Tuple[float, str]:
    """Call the judge model and extract a numeric score in [0, 1]."""
    raw = call_groq(
        system_prompt=_EVAL_SYSTEM_PROMPT,
        user_message=prompt,
        model=eval_model,
        max_tokens=EVAL_MAX_TOKENS,
        temperature=0.0,
    )
    if raw is None:
        return 0.0, "API error"
    try:
        cleaned = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`")
        match   = re.search(r'\{.*\}', cleaned, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
            score  = float(parsed.get("score", 0.0))
            reason = str(parsed.get("reason", ""))
            return max(0.0, min(1.0, score)), reason
    except (json.JSONDecodeError, ValueError, TypeError):
        pass
    # Fallback: try to extract a bare float from the raw response
    nums = re.findall(r'\b0\.\d+\b|\b1\.0\b|\b[01]\b', raw)
    if nums:
        return max(0.0, min(1.0, float(nums[0]))), raw[:80]
    return 0.0, "Parse error"


# Select evaluation subset (stratified by true label for representative scoring)
if EVAL_SUBSET_SIZE is not None and EVAL_SUBSET_SIZE < len(results_df):
    n_each      = EVAL_SUBSET_SIZE // 2
    eval_normal = (
        results_df[results_df["true_label"] == 0]
        .sample(n=min(n_each, (results_df["true_label"] == 0).sum()),
                random_state=RANDOM_SEED)
    )
    eval_anom = (
        results_df[results_df["true_label"] == 1]
        .sample(n=min(n_each, (results_df["true_label"] == 1).sum()),
                random_state=RANDOM_SEED)
    )
    eval_df = pd.concat([eval_normal, eval_anom]).reset_index(drop=True)
    print(f"\nLLM-as-judge evaluation on {len(eval_df)} stratified test cases ...")
else:
    eval_df = results_df.reset_index(drop=True)
    print(f"\nLLM-as-judge evaluation on all {len(eval_df)} test cases ...")

print(f"  Judge model : {EVAL_MODEL}")

eval_scores: List[Dict[str, float]] = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="LLM-as-Judge"):
    log_text    = row["log_text"]
    explanation = row.get("explanation", {})
    if not isinstance(explanation, dict):
        try:
            explanation = json.loads(explanation) if explanation else {}
        except (json.JSONDecodeError, TypeError):
            explanation = {}

    full_response = {
        "label":       row.get("label"),
        "confidence":  row.get("confidence"),
        "explanation": explanation,
    }

    faith_s,    _ = score_with_judge(_build_faithfulness_prompt(log_text, explanation))
    time.sleep(REQUEST_DELAY_S)
    relev_s,    _ = score_with_judge(_build_answer_relevance_prompt(log_text, full_response))
    time.sleep(REQUEST_DELAY_S)
    sre_s,      _ = score_with_judge(_build_role_appropriateness_prompt(full_response, "SRE"))
    time.sleep(REQUEST_DELAY_S)
    devops_s,   _ = score_with_judge(_build_role_appropriateness_prompt(full_response, "DevOps"))
    time.sleep(REQUEST_DELAY_S)
    complete_s, _ = score_with_judge(_build_completeness_prompt(full_response))
    time.sleep(REQUEST_DELAY_S)

    eval_scores.append({
        "faithfulness":                faith_s,
        "answer_relevance":            relev_s,
        "role_appropriateness_sre":    sre_s,
        "role_appropriateness_devops": devops_s,
        "completeness":                complete_s,
    })

eval_scores_df = pd.DataFrame(eval_scores)
mean_scores    = eval_scores_df.mean()

print("\n" + "=" * 64)
print(f"  LLM-AS-JUDGE METRICS  [{DATASET}]  (n={len(eval_df)})")
print("=" * 64)
print(f"  Faithfulness                   : {mean_scores['faithfulness']:.4f}")
print(f"  Answer Relevance               : {mean_scores['answer_relevance']:.4f}")
print(f"  Role Appropriateness (SRE)     : {mean_scores['role_appropriateness_sre']:.4f}")
print(f"  Role Appropriateness (DevOps)  : {mean_scores['role_appropriateness_devops']:.4f}")
print(f"  Completeness                   : {mean_scores['completeness']:.4f}")
print("=" * 64)

metrics.update({
    "faithfulness":                float(mean_scores["faithfulness"]),
    "answer_relevance":            float(mean_scores["answer_relevance"]),
    "role_appropriateness_sre":    float(mean_scores["role_appropriateness_sre"]),
    "role_appropriateness_devops": float(mean_scores["role_appropriateness_devops"]),
    "completeness":                float(mean_scores["completeness"]),
})

# =============================================================================
# SECTION 10 — VISUALISATIONS
#
# Panel layout mirrors E01_C with LLM-specific adaptations:
#   10a — LLM confidence score distribution (Normal vs Anomalous)
#   10b — Confusion matrix
#   10c — Precision-Recall curve (confidence as ranking signal)
#   10d — ROC curve with AUC
#   10e — LLM-as-judge metrics bar chart
#   10f — Detection metrics summary bar chart
# =============================================================================

def plot_results(
    conf_scores: List[float],
    y_true: List[int],
    y_pred: List[int],
    pr_precisions: np.ndarray,
    pr_recalls: np.ndarray,
    fpr_roc: np.ndarray,
    tpr_roc: np.ndarray,
    auroc: float,
    cm: np.ndarray,
    metrics: Dict[str, Any],
    eval_scores_df: pd.DataFrame,
    dataset_name: str,
    save_prefix: Optional[str] = None,
) -> None:
    """Render and save six diagnostic plots for the LLM baseline."""
    prefix   = save_prefix or f"{dataset_name.lower()}_llm"
    conf_arr = np.array(conf_scores)
    y_arr    = np.array(y_true)

    # 10a — Confidence score distribution
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(
        conf_arr[y_arr == 0], bins=20, alpha=0.6, color="steelblue",
        label=f"Normal (n={(y_arr == 0).sum()})", edgecolor="white",
    )
    ax.hist(
        conf_arr[y_arr == 1], bins=20, alpha=0.6, color="crimson",
        label=f"Anomalous (n={(y_arr == 1).sum()})", edgecolor="white",
    )
    ax.axvline(
        0.5, color="black", linestyle="--", lw=2,
        label="Decision boundary (confidence = 0.5)",
    )
    ax.set_xlabel("LLM Anomaly Confidence Score")
    ax.set_ylabel("Count")
    ax.set_title(
        f"LLM Confidence Score Distribution — {dataset_name}",
        fontweight="bold",
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fname = f"{prefix}_confidence_dist.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")

    # 10b — Confusion matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Predicted Normal", "Predicted Anomalous"],
        yticklabels=["Actual Normal",    "Actual Anomalous"],
        linewidths=0.5, linecolor="gray",
        annot_kws={"size": 14, "weight": "bold"}, ax=ax,
    )
    for (r, c), lbl in {(0, 0): "TN", (0, 1): "FP", (1, 0): "FN", (1, 1): "TP"}.items():
        ax.text(
            c + 0.5, r + 0.72, lbl,
            ha="center", va="center", fontsize=10, color="grey",
        )
    ax.set_title(
        f"Confusion Matrix — LLM Few-Shot+CoT [{dataset_name}]",
        fontweight="bold", pad=14,
    )
    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    fname = f"{prefix}_confusion_matrix.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")

    # 10c — Precision-Recall curve
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(pr_recalls, pr_precisions, color="darkorange", lw=2, label="PR Curve")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(
        f"Precision-Recall Curve — LLM Few-Shot+CoT [{dataset_name}]",
        fontweight="bold",
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fname = f"{prefix}_pr_curve.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")

    # 10d — ROC curve
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(
        fpr_roc, tpr_roc, color="darkorchid", lw=2,
        label=f"ROC Curve (AUC = {auroc:.4f})",
    )
    ax.plot(
        [0, 1], [0, 1], color="grey", linestyle="--", lw=1,
        label="Random classifier (AUC = 0.50)",
    )
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate (Recall)")
    ax.set_title(
        f"ROC Curve — LLM Few-Shot+CoT [{dataset_name}]  |  AUC = {auroc:.4f}",
        fontweight="bold",
    )
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fname = f"{prefix}_roc_curve.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")

    # 10e — LLM-as-judge evaluation metrics bar chart
    judge_names = [
        "Faithfulness",
        "Answer\nRelevance",
        "Role Appr.\n(SRE)",
        "Role Appr.\n(DevOps)",
        "Completeness",
    ]
    judge_values = [
        metrics.get("faithfulness", 0),
        metrics.get("answer_relevance", 0),
        metrics.get("role_appropriateness_sre", 0),
        metrics.get("role_appropriateness_devops", 0),
        metrics.get("completeness", 0),
    ]
    colours = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(
        judge_names, judge_values,
        color=colours, edgecolor="white", alpha=0.85,
    )
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score (0 – 1)")
    ax.set_title(
        f"LLM-as-Judge Evaluation Metrics — {LLM_MODEL} [{dataset_name}]",
        fontweight="bold",
    )
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fname = f"{prefix}_llm_eval_metrics.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")

    # 10f — Detection metrics summary bar chart
    det_names  = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"]
    det_values = [
        metrics["accuracy"],
        metrics["precision"],
        metrics["recall"],
        metrics["f1"],
        metrics["auroc"],
    ]
    fig, ax = plt.subplots(figsize=(9, 4))
    bars2 = ax.bar(
        det_names, det_values,
        color="steelblue", edgecolor="white", alpha=0.85,
    )
    ax.bar_label(bars2, fmt="%.4f", padding=3, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title(
        f"Detection Metrics Summary — LLM Few-Shot+CoT [{dataset_name}]",
        fontweight="bold",
    )
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fname = f"{prefix}_detection_metrics.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


print("\nGenerating visualisations ...")
plot_results(
    conf_scores=conf_scores,
    y_true=y_true,
    y_pred=y_pred,
    pr_precisions=pr_precisions,
    pr_recalls=pr_recalls,
    fpr_roc=fpr_roc,
    tpr_roc=tpr_roc,
    auroc=auroc,
    cm=metrics["cm"],
    metrics=metrics,
    eval_scores_df=eval_scores_df,
    dataset_name=DATASET,
)

  Dataset          : BGL
  LLM model        : llama-3.3-70b-versatile
  Eval model       : qwen/qwen3-32b
  Few-shot normal  : 3
  Few-shot anomaly : 3
  Eval subset size : 100
  Train ratio      : 0.6
  Val ratio        : 0.2
  Test normals     : remaining 20% of normals
  Test anomalies   : ALL available (minus few-shot pool)
  Groq key source  : key file
Loading dataset: BGL
  Loading BGL structured log ...
    Rows: 2000, Templates: 120

  Total samples : 2000
  Normal        : 1857  (92.8%)
  Anomalous     : 143  (7.1%)

  Sample log_text:
    [KERNEL] [INFO] instruction cache parity error corrected | Template: instruction cache parity error corrected
Creating train / validation / test splits ...

  Train  :   1114  normal only
  Val    :    371  normal only
  Test   :    512  total
              372  normal     (72.7%)
              140  anomalous  (27.3%)
  Few-shot anomaly pool : 3 samples (excluded from test evaluation)

Few-shot examples selected:
  Normal    : 3 (from train 

LLM Inference:   4%|▍         | 22/512 [03:05<1:25:49, 10.51s/it]

  API error (attempt 1/3): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jxagj5xxfhabcw1g8gpmgc1z` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99828, Requested 2117. Please try again in 28m0.479999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Retrying in 5s ...
  API error (attempt 2/3): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jxagj5xxfhabcw1g8gpmgc1z` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99822, Requested 2117. Please try again in 27m55.296s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Retrying in 10s ...
  API error (attempt 3/3): Error code: 429 - {'error': {'message': 'Rate limit r

LLM Inference:   4%|▍         | 23/512 [03:35<2:15:02, 16.57s/it]

  API error (attempt 1/3): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jxagj5xxfhabcw1g8gpmgc1z` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99792, Requested 2129. Please try again in 27m39.744s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Retrying in 5s ...
  API error (attempt 2/3): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jxagj5xxfhabcw1g8gpmgc1z` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99786, Requested 2116. Please try again in 27m23.328s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Retrying in 10s ...
  API error (attempt 3/3): Error code: 429 - {'error': {'message': 'Rate limit reache

LLM Inference:   4%|▍         | 23/512 [04:06<1:27:11, 10.70s/it]


KeyboardInterrupt: 